> 本 notebook 由 AI 翻译自英文原文，可能存在疏漏。以准确性为准时请参阅[英文原版](../../../../02-explore-agentic-frameworks/code_samples/02-python-langchain-agent.ipynb)。代码单元与英文版完全相同，只翻译了说明文字。

# 第 02 课 - 探索智能体框架

智能体框架提供了一小组可组合的积木，让你不必自己编写模型调用循环、工具分发代码和对话记录的维护逻辑。在 LangChain 和 LangGraph 里，这些积木是：

- **模型客户端（Model client）**——连接 AI 模型接口并负责通信
- **智能体（Agent）**——把模型客户端、指令和工具定义包在一起，运行工具调用循环
- **工具（Tools）**——用模型可以调用的自定义函数扩展智能体的能力
- **记忆（线程）**——一个检查点器（checkpointer），保存对话历史，让多轮对话得以进行

本课我们会用这些概念构建一个**旅行预订智能体**，用来查询目的地是否可预订。

## 环境准备

前置条件：在仓库根目录运行 `pip install -r requirements.txt`，把 `.env.example` 复制为 `.env`，填入 `LLM_BASE_URL`、`LLM_API_KEY`、`LLM_MODEL`，然后运行 `python scripts/check_endpoint.py`。

下面的单元会从 `.env` 加载这些变量并构建聊天模型客户端。`ChatOpenAI` 使用 OpenAI Chat Completions 协议，所以同一段代码可以对接 DeepSeek、OpenAI、本地 Ollama 服务或任何其他兼容接口——只需要改这三个环境变量。`LLM_EXTRA_BODY` 用来传递可选的、特定于服务商的请求选项（对 DeepSeek 来说，它会关闭思考模式）。

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

missing = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL") if not os.environ.get(name)]
if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Copy .env.example to .env in the repository root and fill them in."
    )

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    extra_body=json.loads(os.environ.get("LLM_EXTRA_BODY") or "null"),
)
print(f"Model client ready: {os.environ['LLM_MODEL']} @ {os.environ['LLM_BASE_URL']}")

Model client ready: deepseek-v4-pro @ https://api.deepseek.com/v1


## 理解框架架构

这些组件按层次组合在一起：

```
ChatOpenAI  →  create_agent  →  tools
                             →  checkpointer (thread)
```

1. **模型客户端**——`ChatOpenAI` 只负责 HTTP 通信。它按 Chat Completions API 的格式组装请求，发送到 `LLM_BASE_URL`，再解析回复（包括模型请求的任何工具调用）。它对循环和对话一无所知。
2. **智能体**——`create_agent` 掌管**工具调用循环**：把对话发给模型，执行模型请求的工具，把结果追加进去，如此反复，直到模型以纯文本作答。你传入的指令会成为系统提示词。
3. **工具**——用 `@tool` 装饰的普通 Python 函数。智能体把根据函数签名和 docstring 生成的 schema 交给模型，并在模型请求时调用该函数。
4. **记忆（线程）**——**检查点器**在每一步之后保存消息历史。每个对话由一个 `thread_id` 标识；传同一个 id，智能体就记得之前的轮次；传一个新的 id，就从头开始。

下面我们一层一层地构建。模型客户端已经在上面的环境准备单元里创建好了。

## 用 @tool 装饰器添加工具

工具让智能体能做生成文本之外的事。`@tool` 装饰器把一个普通的 Python 函数变成智能体可以调用的东西。

要点：
- **docstring** 会成为模型看到的工具描述。
- **类型注解**定义了参数的 schema。
- `Annotated[type, "description"]` 为每个参数添加描述，让模型明白该传什么。
- 模型请求时工具会自动运行；后面的课程会展示如何先经过人工批准。

In [2]:
from typing import Annotated

from langchain.tools import tool


@tool
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 创建带工具的智能体

现在把模型客户端、指令和工具组合成一个智能体。`system_prompt` 定义了智能体的角色和行为方式。

我们还挂上了一个 `InMemorySaver` **检查点器**。没有它，智能体在两次 `invoke` 调用之间会忘掉一切；有了它，每个由 `thread_id` 标识的对话都会被保存下来，并在下一轮回放。

`agent.invoke` 返回完整的消息历史——用户消息、模型发起的工具调用、工具结果，以及最终回复。`reply_text` 从最后一条消息里取出文本（有些服务商返回的 content 是一组内容块的列表，而不是单个字符串）。

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    llm,
    tools=[check_destination_availability],
    system_prompt=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    checkpointer=InMemorySaver(),
)


def reply_text(result) -> str:
    """Return the text of the last message; content can be a string or a list of blocks."""
    content = result["messages"][-1].content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return content

## 用线程进行多轮对话

**线程（thread）**是检查点器记住的一段对话。你通过一个配置字典 `{"configurable": {"thread_id": "..."}}` 来选择它，作为第二个参数传给 `invoke`。每次调用都传同一个配置，智能体就能访问完整的对话历史，并回溯之前的消息。

智能体在任何一轮都可以调用 `check_destination_availability`——工具调用循环在每次 `invoke` 内部都会运行。

这个工具只回答你*点名*的目的地；做完本 notebook 之后，添加一个 `list_destinations` 工具是个不错的练习。

In [4]:
session = {"configurable": {"thread_id": "travel-availability-1"}}

# Turn 1: Ask about specific destinations
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Which of Barcelona, Tokyo, Cape Town, Vancouver and Dubai are available?"}]},
    session,
)
print(f"Agent: {reply_text(result)}")

# Turn 2: Follow-up question — the agent remembers the conversation
result = agent.invoke(
    {"messages": [{"role": "user", "content": "I'd like to go somewhere warm. What's available?"}]},
    session,
)
print(f"\nAgent: {reply_text(result)}")

Agent: Here are the results:

**Available for booking:**
- ✅ Barcelona
- ✅ Tokyo
- ✅ Vancouver

**Not available for booking:**
- ❌ Cape Town
- ❌ Dubai

Would you like any recommendations or help planning a trip to one of the available destinations?

Agent: Based on your preference for warm weather, here's how the available destinations stack up:

**Available warm destinations:**
- ✅ **Barcelona** – Mediterranean climate with warm, sunny weather, especially a great choice for beach and culture.
- ✅ **Vancouver** – typically mild, though it's better known for a temperate (rather than hot) climate; warmth depends on the season.

**Tokyo** is also available but tends to have more distinct seasons, with hot and humid summers but cooler winters.

**Cape Town and Dubai** (both unavailable) would have been strong warm-weather options, but they're not currently bookable.

If you're looking for the warmest option right now, **Barcelona** would likely be your best bet. Would you like me to help y

## 第二个工具与流式输出

工具可以拥有自己的状态。下面的工具随机挑选一个度假目的地，并记住上一次的选择，这样连续两次调用永远不会返回同一个地方——当用户否决了一个建议、要求再来一个时，这很有用。

我们围绕它创建第二个智能体，同样带检查点器，这次以**流式**方式接收回复。`agent.astream(..., stream_mode="messages")` 会为图产生的每一个消息块产出一对 `(token, metadata)`；我们打印来自模型节点的块，并在工具节点运行时另起一行，这样工具调用前后的文本就不会连在一起。Jupyter 支持顶层 `await`，所以下面的 `async for` 可以直接运行。

两轮共用一个 `thread_id`，因此在第二轮时智能体知道哪个目的地被否决了。

In [5]:
import random

# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


random_trip_agent = create_agent(
    llm,
    tools=[get_random_destination],
    system_prompt="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    checkpointer=InMemorySaver(),
)

user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]
session = {"configurable": {"thread_id": "random-trip-1"}}

for turn, user_input in enumerate(user_inputs, start=1):
    print(f"--- turn {turn} ---")
    print(f"User: {user_input}\nAgent: ", end="")
    async for token, metadata in random_trip_agent.astream(
        {"messages": [{"role": "user", "content": user_input}]}, session, stream_mode="messages"
    ):
        if metadata.get("langgraph_node") == "model" and getattr(token, "content", None):
            print(token.content, end="", flush=True)
        elif metadata.get("langgraph_node") == "tools":
            print()  # blank line between the text before and after a tool call
    print("\n")

--- turn 1 ---
User: Plan me a day trip.
Agent: I'll find a random destination for your day trip!
# 🌍 Day Trip: Cape Town, South Africa

Here's a packed (but relaxed) one-day itinerary for Cape Town:

## 🏔️ Morning — Iconic Landmarks
- **Sunrise at Table Mountain** – Take the Aerial Cableway up for panoramic views of the city, ocean, and Table Bay. (Go early to beat the crowds and the clouds.)
- **Breakfast** in the V&A Waterfront area – grab a coffee and pastry with harbor views.

## 🐧 Midday — Coastal & Culture
- **Bo-Kaap neighborhood** – a quick stroll through the colorful houses, perfect for photos.
- **Chapman's Peak Drive** – one of the world's most scenic coastal drives (pull over for viewpoints).
- **Lunch** at **Hout Bay** – try fresh fish & chips at the harbor market.

## 🐧 Afternoon — Wildlife & Nature
- **Boulders Beach** – see the famous African penguin colony wandering the beach.
- Optional: visit **Cape Point** & the Cape of Good Hope for dramatic cliffs and coastline.


## 小结

本课你探索了智能体框架的四块积木：

| 概念 | 你学到了什么 |
|---------|------------------|
| **模型客户端** | `ChatOpenAI` 可以对接任何 OpenAI 兼容接口；服务商只是一个配置项 |
| **智能体** | `create_agent` 把模型客户端、指令和工具打包在一起，运行工具调用循环 |
| **工具** | `@tool` 装饰器把 Python 函数（docstring + 类型注解）暴露给智能体调用 |
| **记忆** | `InMemorySaver` 加上 `thread_id`，在多轮对话之间维护对话历史 |

这些积木组合起来，就能创建出能自然对话、调用外部函数并保持上下文的智能体——这是后续课程中更高级智能体模式的基础。